# Lesson 01 - Въведение в AI агенти

Добре дошли в първия урок от курса **AI агенти за начинаещи**!

**AI агент** е програма, която използва голям езиков модел (LLM) като своя логически двигател и може да предприема *действия* в реалния свят — да извиква API-та, да прави заявки към бази данни или да изпълнява код — с цел постигане на задача от името на потребителя.

В тази тетрадка ще създадете своя първи агент: **Туристически агент**, който препоръчва ваканционни дестинации. По пътя ще научите как да:

1. Свържете се с Azure AI Foundry Agent Service чрез **Microsoft Agent Framework**.
2. Предоставите на агента **инструмент** — обикновена Python функция, която може да извиква.
3. Стартирате агента и инспектирате отговора му.
4. Поточно получавате отговора на агента токен по токен.


## Настройка

Преди да стартирате този ноутбук, уверете се, че имате:

1. **Проект в Azure AI Foundry** с разположен чат модел (напр. `gpt-4o-mini`).
2. **Влезли сте с Azure CLI** — изпълнете `az login` в терминала си.
3. **Зададени са необходимите променливи на средата:**
   - `AZURE_AI_PROJECT_ENDPOINT` — вашият крайна точка на проекта в Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — името на вашия разгърнат модел.

По-долу празната клетка инсталира необходимите Python пакети.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Създаване на Вашия Първи Агент

Агентът се нуждае от две неща:

- **Инструкции**, които му казват *кой е той* и *как да се държи* (системно подканяне).
- **Инструменти** — Python функции, декорирани с `@tool`, които агентът може да извиква, за да получи информация или да изпълнява действия.

По-долу дефинираме прост инструмент, който връща списък с популярни ваканционни дестинации. Агентът ще използва този инструмент, когато потребителят поиска препоръки за пътувания.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Поточно предаване на отговори

За по-интерактивно изживяване можете да **поточно предавате** отговора на агента. Вместо да чакате пълния отговор, агентът предоставя текстови части веднага щом бъдат генерирани. Това е особено полезно в чат интерфейси, където искате да показвате резултата в реално време.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резюме

В този урок научихте как да:

- **Създадете доставчик**, който се свързва с Azure AI Foundry Agent Service чрез `AzureAIProjectAgentProvider`.
- **Дефинирате инструмент** с помощта на декоратора `@tool`, за да може агентът да извиква вашите Python функции.
- **Стартирате агента** с потребителско съобщение и отпечатате неговия отговор.
- **Изпращате поточно отговори** за изход в реално време.

В следващия урок ще разгледаме по-задълбочено агентските рамки и ще научим как да дадем на агентите по-мощни инструменти и възможности за многостоенчески разсъждения.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматичните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
